# Merchant Ranking System

This notebook develops an interpretable ranking system for selecting merchants for BNPL onboarding.

The ranking framework will combine merchant value, customer strength, growth, stability, market context, and fraud risk.

The current version first develops the non-fraud components. Fraud-related features will be integrated once the fraud module is completed.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

INPUT_PATH = (
    PROJECT_ROOT
    / "member3_merchant_features"
    / "results"
    / "merchant_features.parquet"
)

df = pd.read_parquet(INPUT_PATH)

print("Shape:", df.shape)
df.head()

Shape: (4422, 46)


,merchant_abn,merchant_name,merchant_category,merchant_pricing_level,merchant_take_rate_pct,has_merchant_master_record,total_transactions,total_revenue,avg_transaction_value,unique_consumers,...,census_single_parent_family_share,census_couple_with_children_family_share,census_unemployment_rate_published_pct,census_labour_force_participation_rate_published_pct,seifa_irsd_national_decile,seifa_irsad_national_decile,seifa_ier_national_decile,ato_taxable_income_or_loss_per_reporter,ato_salary_or_wages_per_recipient,ato_net_tax_payer_share
0,10023283211,Felis Limited,"furniture, home furnishings and equipment shop...",E,0.18,True,3261,703277.711451,215.663205,3032,...,0.137732,0.396714,4.545196,59.159350,5.426345,5.412987,5.455659,68260.371739,63998.111541,0.734878
1,10142254217,Arcu Ac Orci Corporation,"cable, satellite, and other pay television and...",B,4.22,True,3036,118356.146073,38.984238,2849,...,0.137717,0.397312,4.618711,59.135731,5.512892,5.524395,5.500595,69361.310505,64617.995761,0.734825
2,10165489824,Nunc Sed Company,"jewelry, watch, clock, and silverware shops",B,4.40,True,5,56180.473857,11236.094771,5,...,0.114423,0.490410,4.150000,63.625000,8.250000,8.000000,8.500000,76144.193842,63913.702304,0.782871
3,10187291046,Ultricies Dignissim Lacus Foundation,"watch, clock, and jewelry repair shops",B,3.29,True,336,39693.730387,118.136102,335,...,0.132108,0.393778,4.485274,59.633219,5.513793,5.434483,5.548276,68884.847450,65123.944932,0.734622
4,10192359162,Enim Condimentum PC,"music shops - musical instruments, pianos, and...",A,6.33,True,385,177980.505456,462.287027,383,...,0.136463,0.401961,4.413313,60.091331,5.744548,5.679128,5.732087,70416.453471,66239.804559,0.746924


## 1. Candidate Pool

The merchant feature table contains 4,422 merchants.

For the preliminary ranking, only merchants with a valid merchant master record are treated as eligible candidates.

Transaction-only orphan merchants are excluded because merchant category and take-rate information are unavailable, which prevents fair comparison on key BNPL value metrics.

In [2]:
eligible_df = df[df["has_merchant_master_record"]].copy()

print("All merchants:", len(df))
print("Eligible merchants:", len(eligible_df))
print("Excluded orphan merchants:", len(df) - len(eligible_df))

All merchants: 4422
Eligible merchants: 4026
Excluded orphan merchants: 396


## 2. Candidate Ranking Features

The preliminary ranking uses merchant-level features that capture business value, customer strength, growth, stability and market context.

Fraud-related features will be integrated after the fraud module is completed.

In [3]:
candidate_features = [
    "estimated_bnpl_revenue",
    "total_revenue",
    "total_transactions",
    "unique_consumers",
    "avg_transaction_value",
    "repeat_consumer_share",
    "normalized_monthly_revenue_trend",
    "monthly_revenue_cv",
    "low_sample_growth_estimate",
    "regional_data_coverage_rate_all_sources_by_count",
    "census_median_household_income_weekly",
    "seifa_irsad_national_decile",
    "ato_taxable_income_or_loss_per_reporter",
]

eligible_df[candidate_features].describe().T

,count,mean,std,min,25%,50%,75%,max
estimated_bnpl_revenue,4026.0,23933.339418,5.849947e+04,12.313488,1470.210886,5146.420842,22933.624931,6.638703e+05
total_revenue,4026.0,535781.369254,1.218831e+06,10064.933919,36652.804201,141397.947807,628006.258580,9.857402e+06
total_transactions,4026.0,3381.687779,1.414670e+04,1.000000,93.000000,418.000000,2057.500000,2.895130e+05
unique_consumers,4026.0,2025.218579,3.875050e+03,1.000000,93.000000,415.500000,1961.000000,2.408100e+04
avg_transaction_value,4026.0,1159.595793,2.947518e+03,7.588294,118.099522,317.329914,836.237711,5.187664e+04
repeat_consumer_share,4026.0,0.051295,1.185795e-01,0.000000,0.000000,0.009718,0.041562,9.999169e-01
normalized_monthly_revenue_trend,4019.0,0.011004,4.624178e-02,-0.857143,0.006984,0.014355,0.020337,6.551908e-01
monthly_revenue_cv,4019.0,0.482603,4.134482e-01,0.133836,0.217327,0.321069,0.577596,3.872983e+00
regional_data_coverage_rate_all_sources_by_count,4026.0,0.806718,5.417902e-02,0.000000,0.796875,0.807871,0.819749,1.000000e+00
census_median_household_income_weekly,4026.0,1603.516538,8.298170e+01,806.500000,1587.309716,1604.879927,1620.820699,3.124000e+03


In [4]:
feature_missingness = (
    eligible_df[candidate_features]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("missing_rate")
)

feature_missingness

,missing_rate
normalized_monthly_revenue_trend,0.001739
monthly_revenue_cv,0.001739
ato_taxable_income_or_loss_per_reporter,0.000248
estimated_bnpl_revenue,0.000000
total_revenue,0.000000
total_transactions,0.000000
unique_consumers,0.000000
avg_transaction_value,0.000000
repeat_consumer_share,0.000000
low_sample_growth_estimate,0.000000


## 3. Feature Redundancy Check

Before assigning weights, pairwise correlations are examined to reduce double-counting of closely related merchant characteristics.

In [5]:
correlation_features = [
    "estimated_bnpl_revenue",
    "total_revenue",
    "total_transactions",
    "unique_consumers",
    "avg_transaction_value",
    "repeat_consumer_share",
    "normalized_monthly_revenue_trend",
    "monthly_revenue_cv",
    "census_median_household_income_weekly",
    "seifa_irsad_national_decile",
    "ato_taxable_income_or_loss_per_reporter",
]

eligible_df[correlation_features].corr().round(2)

,estimated_bnpl_revenue,total_revenue,total_transactions,unique_consumers,avg_transaction_value,repeat_consumer_share,normalized_monthly_revenue_trend,monthly_revenue_cv,census_median_household_income_weekly,seifa_irsad_national_decile,ato_taxable_income_or_loss_per_reporter
estimated_bnpl_revenue,1.00,0.92,0.56,0.61,-0.03,0.64,0.03,-0.24,0.01,0.00,0.00
total_revenue,0.92,1.00,0.57,0.65,-0.02,0.66,0.03,-0.25,0.01,0.00,0.00
total_transactions,0.56,0.57,1.00,0.71,-0.08,0.82,0.02,-0.16,0.01,0.00,0.00
unique_consumers,0.61,0.65,0.71,1.00,-0.17,0.98,0.04,-0.35,0.01,0.01,0.01
avg_transaction_value,-0.03,-0.02,-0.08,-0.17,1.00,-0.14,-0.21,0.55,-0.03,-0.00,-0.09
repeat_consumer_share,0.64,0.66,0.82,0.98,-0.14,1.00,0.04,-0.29,0.01,0.01,0.01
normalized_monthly_revenue_trend,0.03,0.03,0.02,0.04,-0.21,0.04,1.00,-0.22,0.22,0.14,0.13
monthly_revenue_cv,-0.24,-0.25,-0.16,-0.35,0.55,-0.29,-0.22,1.00,-0.04,-0.02,-0.03
census_median_household_income_weekly,0.01,0.01,0.01,0.01,-0.03,0.01,0.22,-0.04,1.00,0.75,0.65
seifa_irsad_national_decile,0.00,0.00,0.00,0.01,-0.00,0.01,0.14,-0.02,0.75,1.00,0.52


In [6]:
correlation_matrix = eligible_df[correlation_features].corr().round(3)

correlation_matrix.to_csv(
    PROJECT_ROOT / "member5_ranking" / "results" / "feature_correlation_matrix.csv"
)

correlation_matrix

,estimated_bnpl_revenue,total_revenue,total_transactions,unique_consumers,avg_transaction_value,repeat_consumer_share,normalized_monthly_revenue_trend,monthly_revenue_cv,census_median_household_income_weekly,seifa_irsad_national_decile,ato_taxable_income_or_loss_per_reporter
estimated_bnpl_revenue,1.000,0.923,0.555,0.613,-0.026,0.636,0.031,-0.239,0.007,0.003,0.004
total_revenue,0.923,1.000,0.569,0.648,-0.022,0.663,0.032,-0.254,0.007,0.003,0.003
total_transactions,0.555,0.569,1.000,0.714,-0.081,0.820,0.020,-0.164,0.005,0.003,0.003
unique_consumers,0.613,0.648,0.714,1.000,-0.168,0.977,0.042,-0.347,0.011,0.006,0.006
avg_transaction_value,-0.026,-0.022,-0.081,-0.168,1.000,-0.142,-0.214,0.546,-0.026,-0.000,-0.085
repeat_consumer_share,0.636,0.663,0.820,0.977,-0.142,1.000,0.035,-0.292,0.010,0.005,0.005
normalized_monthly_revenue_trend,0.031,0.032,0.020,0.042,-0.214,0.035,1.000,-0.223,0.216,0.141,0.132
monthly_revenue_cv,-0.239,-0.254,-0.164,-0.347,0.546,-0.292,-0.223,1.000,-0.045,-0.024,-0.028
census_median_household_income_weekly,0.007,0.007,0.005,0.011,-0.026,0.010,0.216,-0.045,1.000,0.751,0.654
seifa_irsad_national_decile,0.003,0.003,0.003,0.006,-0.000,0.005,0.141,-0.024,0.751,1.000,0.519


## 4. Final Preliminary Feature Set

Highly correlated features were not simultaneously included in the preliminary score to reduce double-counting.

The selected non-fraud features are:

- **BNPL Value:** `estimated_bnpl_revenue`
- **Customer Strength:** `unique_consumers`
- **Growth:** `normalized_monthly_revenue_trend`
- **Stability:** `monthly_revenue_cv`
- **Market Context:** `seifa_irsad_national_decile`

Fraud-related risk features will be added once the fraud module is completed.

## 5. Percentile Normalisation

To combine features measured on different scales, each selected metric is converted to a percentile score from 0 to 100.

Higher percentile scores indicate stronger merchant performance.

For metrics where lower values are preferred, such as revenue volatility, the percentile score is reversed.

In [7]:
score_df = eligible_df.copy()

# Higher is better
score_df["value_score"] = (
    score_df["estimated_bnpl_revenue"]
    .rank(pct=True, method="average") * 100
)

score_df["customer_score"] = (
    score_df["unique_consumers"]
    .rank(pct=True, method="average") * 100
)

score_df["growth_score"] = (
    score_df["normalized_monthly_revenue_trend"]
    .rank(pct=True, method="average") * 100
)

score_df["market_score"] = (
    score_df["seifa_irsad_national_decile"]
    .rank(pct=True, method="average") * 100
)

# Lower volatility is better
score_df["stability_score"] = (
    100
    - score_df["monthly_revenue_cv"]
    .rank(pct=True, method="average") * 100
)

score_df[
    [
        "value_score",
        "customer_score",
        "growth_score",
        "stability_score",
        "market_score",
    ]
].describe().round(2)

,value_score,customer_score,growth_score,stability_score,market_score
count,4026.00,4026.00,4019.00,4019.00,4026.00
mean,50.01,50.01,50.01,49.99,50.01
std,28.87,28.87,28.87,28.87,28.87
min,0.02,0.10,0.02,0.00,0.02
25%,25.02,25.06,25.02,24.99,25.02
50%,50.01,50.02,50.01,49.99,50.01
75%,75.01,75.00,75.01,74.98,75.01
max,100.00,99.98,100.00,99.98,99.98


## 6. Preliminary Weighted Score

A preliminary balanced weighting scheme is used before fraud risk is integrated.

The current weights prioritise direct BNPL value and customer scale, while still allowing growth, stability and market context to influence the ranking.

In [9]:
# Neutral score for merchants whose growth/stability cannot be estimated
score_df["growth_score"] = score_df["growth_score"].fillna(50)
score_df["stability_score"] = score_df["stability_score"].fillna(50)

In [10]:
PRELIMINARY_WEIGHTS = {
    "value_score": 0.30,
    "customer_score": 0.25,
    "growth_score": 0.20,
    "stability_score": 0.15,
    "market_score": 0.10,
}

score_df["preliminary_score"] = sum(
    weight * score_df[column]
    for column, weight in PRELIMINARY_WEIGHTS.items()
)

score_df["preliminary_rank"] = (
    score_df["preliminary_score"]
    .rank(method="min", ascending=False)
    .astype(int)
)

preliminary_top_20 = (
    score_df
    .sort_values("preliminary_score", ascending=False)
    [
        [
            "preliminary_rank",
            "merchant_abn",
            "merchant_name",
            "merchant_category",
            "preliminary_score",
            "value_score",
            "customer_score",
            "growth_score",
            "stability_score",
            "market_score",
        ]
    ]
    .head(20)
)

preliminary_top_20

,preliminary_rank,merchant_abn,merchant_name,merchant_category,preliminary_score,value_score,customer_score,growth_score,stability_score,market_score
3966,1,90568944804,Diam Eu Dolor LLC,tent and awning shops,87.168863,99.230005,93.169399,75.167952,82.483205,67.014406
1414,2,38090089066,Interdum Feugiat Sed Inc.,"furniture, home furnishings and equipment shop...",86.800764,98.683557,99.254844,61.607365,93.928838,59.711873
869,3,27326652377,Tellus Aenean Corporation,"music shops - musical instruments, pianos, and...",86.253316,99.403875,89.195231,75.267479,86.165713,61.549925
3782,4,86772484982,Posuere Cubilia Curae LLC,"digital goods: books, movies, music",85.785250,95.032290,95.355191,59.442647,98.283155,68.057625
3692,5,84703983173,Amet Consulting,"computer programming , data processing, and in...",85.765124,96.572280,99.006458,68.847972,89.524757,48.435171
4259,6,96680767841,Ornare Limited,motor vehicle supplies and new parts,85.723766,99.850969,98.435171,63.772083,90.097039,48.907104
3772,7,86578477987,Leo In Consulting,"watch, clock, and jewelry repair shops",85.650047,99.925484,99.975161,55.113212,95.172929,53.800298
1956,8,49212265466,Auctor Company,"florists supplies, nursery stock, and flowers",85.552242,99.056135,98.782911,60.014929,94.575765,49.503229
2367,9,57699602880,Tellus Id Institute,"stationery, office supplies and printing and w...",85.490283,97.441629,82.103825,66.857427,97.885046,76.775956
1917,10,48534649627,Dignissim Maecenas Foundation,"opticians, optical goods, and eyeglasses",85.327189,99.975161,99.428713,59.119184,89.997512,51.539990


In [11]:
preliminary_top_20.to_csv(
    PROJECT_ROOT / "member5_ranking" / "results" / "preliminary_top_20.csv",
    index=False
)

For merchants whose growth or stability could not be estimated because of insufficient activity in the fixed analysis window, a neutral percentile score of 50 is assigned. These merchants remain flagged as low-confidence rather than being rewarded or penalised for unavailable estimates.

## 7. Sensitivity Analysis

To test whether the merchant ranking is overly dependent on one specific weighting choice, three reasonable weighting scenarios are compared.

The scenarios represent:
- a balanced business view,
- a stronger emphasis on merchant value,
- a stronger emphasis on future growth.

Ranking robustness is assessed using Top 100 overlap and merchant rank stability.

In [12]:
SCENARIOS = {
    "balanced": {
        "value_score": 0.30,
        "customer_score": 0.25,
        "growth_score": 0.20,
        "stability_score": 0.15,
        "market_score": 0.10,
    },
    "value_focused": {
        "value_score": 0.40,
        "customer_score": 0.25,
        "growth_score": 0.15,
        "stability_score": 0.15,
        "market_score": 0.05,
    },
    "growth_focused": {
        "value_score": 0.25,
        "customer_score": 0.20,
        "growth_score": 0.30,
        "stability_score": 0.15,
        "market_score": 0.10,
    },
}

for scenario_name, weights in SCENARIOS.items():
    score_col = f"{scenario_name}_score"
    rank_col = f"{scenario_name}_rank"

    score_df[score_col] = sum(
        weight * score_df[column]
        for column, weight in weights.items()
    )

    score_df[rank_col] = (
        score_df[score_col]
        .rank(method="min", ascending=False)
        .astype(int)
    )

In [13]:
top100_sets = {}

for scenario_name in SCENARIOS:
    score_col = f"{scenario_name}_score"

    top100_sets[scenario_name] = set(
        score_df
        .nlargest(100, score_col)["merchant_abn"]
    )

overlap_results = []

scenario_names = list(SCENARIOS.keys())

for i in range(len(scenario_names)):
    for j in range(i + 1, len(scenario_names)):

        scenario_a = scenario_names[i]
        scenario_b = scenario_names[j]

        overlap_count = len(
            top100_sets[scenario_a]
            & top100_sets[scenario_b]
        )

        overlap_results.append({
            "scenario_a": scenario_a,
            "scenario_b": scenario_b,
            "top100_overlap_count": overlap_count,
            "top100_overlap_rate": overlap_count / 100,
        })

overlap_df = pd.DataFrame(overlap_results)

overlap_df

,scenario_a,scenario_b,top100_overlap_count,top100_overlap_rate
0,balanced,value_focused,85,0.85
1,balanced,growth_focused,80,0.80
2,value_focused,growth_focused,66,0.66


The Top 100 overlap is relatively high between the balanced scenario and the two alternative scenarios (80–85%), suggesting that the baseline ranking is reasonably robust.

However, the overlap between the value-focused and growth-focused scenarios falls to 66%, indicating that merchant selection is sensitive to whether current commercial value or future growth is prioritised.

In [14]:
rank_columns = [
    "balanced_rank",
    "value_focused_rank",
    "growth_focused_rank",
]

score_df["best_rank"] = score_df[rank_columns].min(axis=1)
score_df["worst_rank"] = score_df[rank_columns].max(axis=1)

score_df["rank_range"] = (
    score_df["worst_rank"]
    - score_df["best_rank"]
)

rank_stability = (
    score_df[
        [
            "merchant_abn",
            "merchant_name",
            "balanced_rank",
            "value_focused_rank",
            "growth_focused_rank",
            "rank_range",
        ]
    ]
    .sort_values("rank_range")
)

rank_stability.head(20)

,merchant_abn,merchant_name,balanced_rank,value_focused_rank,growth_focused_rank,rank_range
4236,96190048310,Penatibus Et Inc.,4023,4023,4023,0
178,13747603419,Fermentum Vel Mauris Institute,4024,4024,4024,0
3034,71475747855,Aliquet Molestie Corporation,4025,4025,4025,0
985,29623808496,Iaculis Odio Nam Foundation,4026,4026,4026,0
4144,93915598279,Molestie Pharetra Nibh LLP,4022,4021,4022,1
373,17507773571,Libero Nec Limited,4020,4019,4020,1
2745,65959377833,Lobortis Augue Ltd,4021,4022,4021,1
3608,82882460979,Sem Molestie Foundation,4011,4010,4011,1
280,15704713883,Faucibus Ut Nulla Ltd,4016,4016,4017,1
3994,91062920626,Ac Inc.,4019,4018,4019,1


In [15]:
top100_stability = (
    score_df[score_df["balanced_rank"] <= 100]
    [
        [
            "merchant_abn",
            "merchant_name",
            "balanced_rank",
            "value_focused_rank",
            "growth_focused_rank",
            "rank_range",
        ]
    ]
    .sort_values("balanced_rank")
)

top100_stability

,merchant_abn,merchant_name,balanced_rank,value_focused_rank,growth_focused_rank,rank_range
3966,90568944804,Diam Eu Dolor LLC,1,4,1,3
1414,38090089066,Interdum Feugiat Sed Inc.,2,1,4,3
869,27326652377,Tellus Aenean Corporation,3,13,2,11
3782,86772484982,Posuere Cubilia Curae LLC,4,28,8,24
3692,84703983173,Amet Consulting,5,8,5,3
...,...,...,...,...,...,...
989,29639699851,Sodales Elit Erat Corporation,96,87,135,48
2696,64732735902,Imperdiet Non Vestibulum Institute,97,138,82,56
3348,77590625261,Sed Diam Foundation,98,76,189,113
1300,35556933338,Semper Cursus Integer Limited,99,90,130,40


In [16]:
top100_stability["rank_range"].describe()

count    100.000000
mean      49.510000
std       32.408533
min        3.000000
25%       25.500000
50%       44.000000
75%       70.250000
max      179.000000
Name: rank_range, dtype: float64

In [17]:
top100_most_sensitive = (
    top100_stability
    .sort_values("rank_range", ascending=False)
    .head(20)
)

top100_most_sensitive

,merchant_abn,merchant_name,balanced_rank,value_focused_rank,growth_focused_rank,rank_range
2226,55069630872,A Nunc Corp.,93,200,21,179
4119,93558142492,Dolor Quisque Inc.,94,56,183,127
1969,49505931725,Suspendisse Ac Associates,71,42,164,122
3348,77590625261,Sed Diam Foundation,98,76,189,113
3085,72472909171,Nullam Consulting,73,43,154,111
4175,94729574738,Scelerisque Corporation,72,115,10,105
2396,58454491168,Diam At Foundation,62,40,140,100
2471,60111071436,Imperdiet Non LLC,81,141,45,96
3479,80324045558,Ipsum Dolor Sit Corporation,84,59,155,96
1128,32361057556,Orci In Consequat Corporation,58,31,125,94


In [18]:
top20_stability = (
    score_df[score_df["balanced_rank"] <= 20]
    [
        [
            "merchant_abn",
            "merchant_name",
            "balanced_rank",
            "value_focused_rank",
            "growth_focused_rank",
            "rank_range",
        ]
    ]
    .sort_values("balanced_rank")
)

top20_stability

,merchant_abn,merchant_name,balanced_rank,value_focused_rank,growth_focused_rank,rank_range
3966,90568944804,Diam Eu Dolor LLC,1,4,1,3
1414,38090089066,Interdum Feugiat Sed Inc.,2,1,4,3
869,27326652377,Tellus Aenean Corporation,3,13,2,11
3782,86772484982,Posuere Cubilia Curae LLC,4,28,8,24
3692,84703983173,Amet Consulting,5,8,5,3
4259,96680767841,Ornare Limited,6,3,9,6
3772,86578477987,Leo In Consulting,7,2,32,30
1956,49212265466,Auctor Company,8,5,14,9
2367,57699602880,Tellus Id Institute,9,52,3,49
1917,48534649627,Dignissim Maecenas Foundation,10,6,25,19


In [19]:
# Number of balanced Top 100 merchants that remain Top 100 in all scenarios
stable_top100_count = (
    (score_df["balanced_rank"] <= 100)
    & (score_df["value_focused_rank"] <= 100)
    & (score_df["growth_focused_rank"] <= 100)
).sum()

print("Balanced Top 100 retained in all scenarios:", stable_top100_count)

Balanced Top 100 retained in all scenarios: 66


In [20]:
# Summary for the balanced Top 20
top20_stability["rank_range"].describe()

count    20.000000
mean     18.250000
std      13.466821
min       3.000000
25%       6.750000
50%      17.000000
75%      26.000000
max      49.000000
Name: rank_range, dtype: float64

### Sensitivity Analysis Interpretation

The balanced ranking is moderately robust to reasonable changes in weighting assumptions.

The Top 100 overlap is 85% between the balanced and value-focused scenarios and 80% between the balanced and growth-focused scenarios. The overlap between the two more extreme scenarios falls to 66%, indicating that merchant selection changes when current commercial value and future growth are prioritised differently.

Among the balanced Top 100, 66 merchants remain in the Top 100 under all three scenarios. These merchants can therefore be treated as a relatively robust core recommendation set.

Exact rank positions are more sensitive than Top 100 membership. The median rank range among the balanced Top 100 is 44 positions, while the median rank range among the balanced Top 20 is only 17 positions.

This indicates that the highest-ranked merchants are generally more stable, while sensitivity is concentrated among merchants closer to the Top 100 selection boundary.